# 01 — Bronze ingestion

Reads the seed CSVs from `lh_bronze/Files/{raw,ref}` and writes Delta tables
to `lh_bronze/Tables/bronze_*`.

- **Idempotent:** every run overwrites the bronze snapshot.
- **Read-only on source files**; writes only inside `lh_bronze`.
- **Audit columns:** `_batch_id`, `_ingested_at_utc`, `_source_file`.

This is the first thin slice of the medallion build. Silver and gold
notebooks come next and read from these bronze tables.

In [ ]:
from datetime import datetime, timezone
from pyspark.sql import functions as F

# Sandbox identifiers. Replace when promoting to another workspace.
WORKSPACE_ID = "355cd573-8b17-430a-b606-c6b9424d36a1"
LAKEHOUSE_ID = "0cde2eb7-937b-49c6-a4ed-b95bd66dc49e"  # lh_bronze
BASE = f"abfss://{WORKSPACE_ID}@onelake.dfs.fabric.microsoft.com/{LAKEHOUSE_ID}"

BATCH_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")

RAW = {
    "production_fact": "Files/raw/production_fact.csv",
    "deferment_fact": "Files/raw/deferment_fact.csv",
    "flowstations_dim": "Files/raw/flowstations_dim.csv",
    "deferment_category_dim": "Files/raw/deferment_category_dim.csv",
    "deferment_subcategory_dim": "Files/raw/deferment_subcategory_dim.csv",
}

REF = {
    "dim_asset": "Files/ref/dim_asset.csv",
    "dim_date": "Files/ref/dim_date.csv",
    "dim_station": "Files/ref/dim_station.csv",
    "dim_well": "Files/ref/dim_well.csv",
    "oil_prices": "Files/ref/oil_prices.csv",
    "owner_map": "Files/ref/owner_map.csv",
}


def ingest(name: str, path: str) -> None:
    src = f"{BASE}/{path}"
    df = (
        spark.read.option("header", True)
        .option("inferSchema", True)
        .csv(src)
        .withColumn("_batch_id", F.lit(BATCH_ID))
        .withColumn("_ingested_at_utc", F.current_timestamp())
        .withColumn("_source_file", F.lit(path))
    )
    target = f"{BASE}/Tables/bronze_{name}"
    (
        df.write.format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .save(target)
    )
    spark.sql(
        f"CREATE TABLE IF NOT EXISTS bronze_{name} "
        f"USING DELTA LOCATION '{target}'"
    )
    print(f"bronze_{name}: {df.count():,} rows")


print(f"Batch: {BATCH_ID}")
for table_name, source_path in {**RAW, **REF}.items():
    ingest(table_name, source_path)
print("Bronze ingestion complete.")

In [ ]:
display(
    spark.sql("SHOW TABLES")
    .filter(F.col("tableName").startswith("bronze_"))
    .orderBy("tableName")
)

## Next

1. **Silver:** conform types, deduplicate `target_oil`, map blank cause IDs,
   reconstruct episodes, and write `silver_*` tables.
2. **Gold:** build the station-day mart, deviation signals, and the daily
   risk table.
3. **Schedule:** orchestrate `01 -> 02 -> 03` in a Data Factory pipeline.